# Laboratorio 3 · Respuestas con fuentes: recuperar, citar y medir por componente

**Objetivo:** construir un índice sobre las políticas de una aerolínea ficticia, responder preguntas de pasajeros con citas y medir por separado si el recuperador encontró la evidencia y si el generador se sostuvo en ella, para decidir qué componente corregir.

Trabajen en el equipo del proyecto y alternen quién ejecuta y quién revisa. El laboratorio se ejecuta en **Google Colab** con una cuenta personal de Google; no necesitas cuenta de GitHub.

El producto compartido se guarda en `labs/lab03-rag/`; cada integrante conserva su reflexión y su historial. Descarguen los archivos antes de cerrar Colab: los archivos de la sesión son temporales. La pregunta individual se responde en Canvas al cierre de la clase del 30/10; producto, reflexión e historial tienen plazo hasta el **martes 03/11/2026 a las 23:59, hora de Chile**.

El asistente que construimos informa lo que dicen las políticas; no decide devoluciones ni compensaciones.

## 1. Preparar el entorno

Guarda una copia del notebook en tu Drive y ejecuta la celda de instalación (tarda uno o dos minutos). Después agrega tu clave de Gemini como `GEMINI_API_KEY` o `GOOGLE_API_KEY` en **Secretos** (icono de llave de Colab) y permite que este notebook la lea. Nunca escribas su valor en una celda, captura o conversación.

En la celda de configuración, `MODE` puede ser `real` (consulta los modelos) o `simulated` (respuestas inventadas para practicar sin clave). Se abre en modo simulado; para la entrega cambia a `real` y vuelve a ejecutar.

**Modelo que responde.** En clase usa el servidor del curso (`curso:course-chat`), con `CURSO_API_KEY` y `CURSO_BASE_URL` en Secretos; el docente entrega la clave del día. Fuera de clase el servidor está apagado: elige `google:gemma-4-26b-a4b-it`, gratis con tu clave de Gemini, o `google:gemini-3.5-flash-lite` si Gemma no responde. Indica en tu entrega cuál usaste.

**Embeddings.** El servidor del curso no los calcula: siempre se usa `gemini-embedding-2` con tu clave de Gemini. El paquete trae ya calculados los vectores del corpus y de las preguntas, con el mismo modelo, para no gastar la cuota en clase (`PRECOMPUTED = True`). Con `False` los calculas tú: unas 80 solicitudes, cerca de un minuto con el límite gratuito de 100 textos por minuto.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # fuera de Colab puede no existir el paquete google
    IN_COLAB = False

if IN_COLAB:
    PACKAGE_REF = "lab03"
    # Para el ensayo docente: un ZIP del paquete subido a la sesión reemplaza la etiqueta.
    LOCAL_PACKAGE = Path("/content/curso-ia-lab03.zip")
    package_source = (f"curso-ia[lab03] @ file://{LOCAL_PACKAGE}" if LOCAL_PACKAGE.exists() else
                      f"curso-ia[lab03] @ git+https://github.com/udd-ia-generativa-2026/curso-ia@{PACKAGE_REF}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", package_source])
    print("Paquete instalado. Continúa con los secretos y la configuración.")
else:
    print("Fuera de Colab se usa el paquete del entorno de desarrollo.")

In [ ]:
import asyncio
import json
import os
from statistics import mean
from time import perf_counter

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from curso_ia import create_model, load_config, save_experiment, status, timed_run
from curso_ia.rag import CourseEmbedding, load_corpus

# Cambia "simulated" por "real" para consultar los modelos con tus claves de Secretos.
MODE = "simulated"  # @param ["simulated", "real"]
os.environ["CURSO_MODE"] = MODE

# Modelo que responde, con la forma proveedor:modelo:
#   "curso:course-chat"             servidor del curso con Gemma 4 12B, solo durante las clases de los viernes;
#                                   requiere los secretos CURSO_API_KEY y CURSO_BASE_URL.
#   "google:gemma-4-26b-a4b-it"     Gemma 4, de pesos abiertos, gratis con tu clave de Gemini; para trabajar fuera de clase.
#   "google:gemini-3.5-flash-lite"  Gemini, gratis con tu clave; segunda opción fuera de clase.
MODEL = "curso:course-chat"  # @param ["curso:course-chat", "google:gemma-4-26b-a4b-it", "google:gemini-3.5-flash-lite"]
os.environ["GENERATION_MODEL"] = MODEL

# Vectores del corpus incluidos en el paquete; con False se calculan con tu clave.
PRECOMPUTED = True  # @param {type: "boolean"}

# Servidor del curso sin Secretos: descomenta y pega la clave del día que entrega el docente.
# os.environ["CURSO_BASE_URL"] = "https://tofu.tail07b663.ts.net/v1"
# os.environ["CURSO_API_KEY"] = ""
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT = ROOT / "labs/lab03-rag"
config = load_config(ROOT)
model_ref = config.models["generation"]
embedding = CourseEmbedding(config.models["embedding"], config, precomputed=PRECOMPUTED)
print("Modo:", config.mode, "· Modelo:", model_ref, "· Embeddings:", config.models["embedding"],
      "· Vectores precalculados:", embedding.precomputed)
display(pd.DataFrame([row for row in status(config) if row["role"] != "comparison"]))

## 2. Conocer el corpus y las preguntas

El corpus son **siete políticas sintéticas** de Pacífico Sur, la aerolínea del laboratorio 2: equipaje, cambios de vuelo, devoluciones, retrasos y cancelaciones, asistencia especial, y check-in y embarque. La política de devoluciones tiene **dos versiones**: la vigente desde el 01/07/2026 y la anterior, derogada, que cambia plazos y cargos. Un sistema que no distingue la vigencia puede responder con una cita literal y aun así equivocarse.

Las **24 preguntas** están escritas como las haría un pasajero. Cada una trae una respuesta de referencia y las **frases literales esperadas**: los pedazos de las políticas que una buena respuesta necesita. Tres preguntas no tienen respuesta en el corpus (q08, q15 y q21) y sirven para medir si el sistema sabe abstenerse. Diez forman el subconjunto de generación, porque generar cuesta cuota y tiempo; la recuperación se mide con las 24.

La referencia la propuso el asistente del curso: revisa algunas preguntas contra las políticas antes de confiar en las métricas, y anota en `REFERENCE_NOTES` lo que no compartas. Nunca la cambies para que coincida con una respuesta del sistema.

In [ ]:
documents, questions = load_corpus()
by_id = {question["id"]: question for question in questions}
REFERENCE_NOTES = ""  # Anota aquí desacuerdos con la referencia propuesta.

display(pd.DataFrame([document.metadata for document in documents]))
display(pd.DataFrame([{
    "id": q["id"], "question": q["question"], "answerable": q["answerable"],
    "evidence_quotes": len(q["expected_evidence"]), "generation_subset": q["in_generation_subset"],
} for q in questions]))

In [ ]:
# Una pregunta con su evidencia esperada; cambia el id para mirar otras.
question = by_id["q17"]
print(question["question"], "\n\nRespuesta de referencia:", question["expected_answer"], "\n")
for evidence in question["expected_evidence"]:
    print(f"- {evidence['doc_id']}: «{evidence['quote']}»")

## 3. De documentos a fragmentos, vectores e índice

Un **fragmento** (*chunk*) es un pedazo de documento que se recupera por separado. `SentenceSplitter` corta cada política en fragmentos de un tamaño máximo en tokens sin partir oraciones, con un **solapamiento** del 10% para que una idea en el borde quede completa en alguno de los dos fragmentos.

Un **embedding** es una lista de números (un vector) que ubica cerca los textos de significado parecido. Para buscar, se calcula el vector de la pregunta y se traen los fragmentos más cercanos. El **índice vectorial** guarda los fragmentos con sus vectores; usamos **Qdrant** en memoria, a través de LlamaIndex.

El índice guarda también **metadatos**: documento, título, vigencia y un identificador legible del fragmento (`pol-equipaje#03`), que usaremos para citar. Los metadatos no se mezclan con el texto que se convierte en vector: sirven para filtrar y para citar.

Construimos un índice por tamaño de fragmento: 200, 400 y 800 tokens. Mira cuántos fragmentos salen de cada uno.

In [ ]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

CHUNK_SIZES = [200, 400, 800]
qdrant = QdrantClient(location=":memory:")  # índice local, sin servidor; desaparece al cerrar la sesión


def build_index(chunk_size):
    """Cortar las políticas, nombrar cada fragmento y guardarlo en Qdrant con su vector."""
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_size // 10)
    nodes = splitter.get_nodes_from_documents(documents)
    counters = {}
    for node in nodes:
        doc_id = node.metadata["doc_id"]
        counters[doc_id] = counters.get(doc_id, 0) + 1
        node.metadata["chunk_id"] = f"{doc_id}#{counters[doc_id]:02d}"
        node.excluded_embed_metadata_keys.append("chunk_id")
        node.excluded_llm_metadata_keys.append("chunk_id")
    store = QdrantVectorStore(client=qdrant, collection_name=f"policies-{chunk_size}")
    index = VectorStoreIndex(nodes, storage_context=StorageContext.from_defaults(vector_store=store),
                             embed_model=embedding)
    return index, nodes


indexes, chunks = {}, {}
for size in CHUNK_SIZES:
    indexes[size], chunks[size] = build_index(size)
    print(f"Fragmentos de {size} tokens: {len(chunks[size])}")
print("Textos enviados al proveedor de embeddings hasta ahora:", embedding.requests)

In [ ]:
# La misma pregunta con fragmentos de 200 y de 800 tokens: el primer resultado de cada índice.
# Una pregunta nueva calcula su vector con tu clave; las del conjunto ya lo traen.
demo = by_id["q02"]["question"]
for size in (200, 800):
    best = indexes[size].as_retriever(similarity_top_k=1).retrieve(demo)[0]
    print(f"──── {size} tokens · {best.node.metadata['chunk_id']} · similitud {best.score:.3f}")
    print(best.node.get_content(), "\n")

## 4. Medir el recuperador sin modelo de lenguaje

Antes de generar, medimos si la búsqueda trae la evidencia. Un fragmento recuperado es **relevante** si contiene alguna de las frases literales esperadas; así la referencia no depende del tamaño del fragmento y no hace falta un juez.

- **recall@k**: qué parte de las frases esperadas llegó entre los *k* primeros fragmentos. 1 significa que llegó toda.
- **Tasa de aciertos** (*hit rate*): en qué parte de las preguntas llegó al menos una frase esperada.
- **MRR** (*mean reciprocal rank*): en qué posición aparece el primer fragmento relevante; vale 1 si es el primero, 0,5 si es el segundo, 0,33 si es el tercero y 0 si no aparece entre los *k*.
- **Derogada en el top-k**: en qué parte de las preguntas se coló un fragmento de la política derogada.

El **filtro de vigencia** usa los metadatos: Qdrant descarta la política derogada antes de comparar vectores. Las tres preguntas sin respuesta no entran en recall, aciertos ni MRR, pero sí en la columna de la derogada.

Barrido corto: tres tamaños por *k* de 3 y 5, con y sin filtro (12 configuraciones). Todo corre en tu sesión: cada pregunta calcula su vector una sola vez.

In [ ]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

CURRENT_ONLY = MetadataFilters(filters=[MetadataFilter(key="status", value="vigente")])


def retrieve(chunk_size, text, k, current_only):
    retriever = indexes[chunk_size].as_retriever(similarity_top_k=k, filters=CURRENT_ONLY if current_only else None)
    return retriever.retrieve(text)


def retrieval_scores(question, retrieved):
    """recall@k, acierto y rango recíproco de una pregunta, comparando frases literales."""
    texts = [result.node.get_content() for result in retrieved]
    quotes = [evidence["quote"] for evidence in question["expected_evidence"]]
    found = [any(quote in text for text in texts) for quote in quotes]
    ranks = [rank for rank, text in enumerate(texts, start=1) if any(quote in text for quote in quotes)]
    return {
        "recall": mean(found) if quotes else None,
        "hit": any(found) if quotes else None,
        "reciprocal_rank": (1 / ranks[0] if ranks else 0.0) if quotes else None,
        "derogated_in_top": any(result.node.metadata["status"] == "derogada" for result in retrieved),
    }


def summarize(rows):
    answerable = [row for row in rows if row["recall"] is not None]
    return {
        "recall_at_k": round(mean(row["recall"] for row in answerable), 2),
        "hit_rate": round(mean(row["hit"] for row in answerable), 2),
        "mrr": round(mean(row["reciprocal_rank"] for row in answerable), 2),
        "derogated_in_top": round(mean(row["derogated_in_top"] for row in rows), 2),
    }


sweep, per_question = [], []
for size in CHUNK_SIZES:
    for current_only in (False, True):
        for k in (3, 5):
            rows = []
            for question in questions:
                rows.append({"id": question["id"], "chunk_size": size, "k": k, "current_only": current_only,
                             **retrieval_scores(question, retrieve(size, question["question"], k, current_only))})
            per_question += rows
            sweep.append({"chunk_size": size, "chunks": len(chunks[size]), "k": k, "current_only": current_only,
                          "rerank": False, **summarize(rows)})

metrics = pd.DataFrame(sweep)
display(metrics)
print("Textos enviados al proveedor de embeddings:", embedding.requests)

## 5. Elegir una configuración

En un corpus tan pequeño el recuperador casi no falla con *k* = 5. La decisión no es «el número más alto», sino un equilibrio entre la evidencia encontrada, el texto que se envía al modelo en cada consulta (*k* × tamaño del fragmento), la vigencia de la fuente y la precisión de la cita: con fragmentos de 800 tokens y *k* = 5 se envía la mitad del corpus.

Elige la configuración que usarán la generación y el rerank, y mira sus fallas pregunta por pregunta: una fila con recall menor que 1 es una **falla de recuperación** candidata para el reporte.

In [ ]:
CHUNK_SIZE = 400  # @param [200, 400, 800]
TOP_K = 5  # @param [3, 5]
CURRENT_ONLY_FILTER = True  # @param {type: "boolean"}

chosen = pd.DataFrame(per_question).query(
    "chunk_size == @CHUNK_SIZE and k == @TOP_K and current_only == @CURRENT_ONLY_FILTER")
print(f"Configuración elegida: fragmentos de {CHUNK_SIZE}, k = {TOP_K}, filtro de vigencia: {CURRENT_ONLY_FILTER}")
display(chosen[(chosen["recall"] < 1) | chosen["derogated_in_top"]]
        .merge(pd.DataFrame(questions)[["id", "question"]], on="id"))

## 6. Responder con citas

Comparamos tres formas de responder la **misma pregunta**, con la misma instrucción y el mismo modelo; solo cambia el contexto:

1. `no_documents`: sin documentos. Sobre una aerolínea ficticia, el modelo solo puede abstenerse o inventar.
2. `full_corpus`: todos los fragmentos del corpus en la consulta, incluida la política derogada. Es posible porque el corpus es pequeño, pero envía todo cada vez.
3. `retrieval`: solo los *k* fragmentos recuperados con la configuración elegida.

La salida es tipada, como en el laboratorio 2: la respuesta, una lista de **citas** (identificador del fragmento y frase copiada literalmente de él) y una marca de **abstención** para cuando la evidencia no alcanza. Corremos el subconjunto de 10 preguntas, que incluye las tres sin respuesta: 30 consultas. En modo simulado las respuestas están preparadas y no se miden tiempos ni tokens.

In [ ]:
class Citation(BaseModel):
    chunk_id: str = Field(description="Identificador del fragmento citado, por ejemplo pol-equipaje#03")
    quote: str = Field(description="Frase copiada literalmente del fragmento que respalda la respuesta")


class Answer(BaseModel):
    answer: str = Field(description="Respuesta al pasajero, breve y en español")
    citations: list[Citation] = Field(description="Una cita por cada afirmación de la respuesta")
    abstained: bool = Field(description="true si los fragmentos no permiten responder")


INSTRUCTION = """
Eres el asistente de atención de Pacífico Sur. Responde la pregunta del pasajero usando solo los fragmentos entregados.
Cada afirmación debe tener una cita con el identificador del fragmento y una frase copiada literalmente de él.
Si los fragmentos no permiten responder, marca abstained=true y no inventes.
""".strip()
SETTINGS = {"temperature": 1.0, "max_tokens": 1500}
MODES = ["no_documents", "full_corpus", "retrieval"]
SIMULATED_ANSWER = {"answer": "Respuesta simulada; no proviene de ningún modelo.", "citations": [], "abstained": True}


def context_for(mode, question):
    """Fragmentos que recibe el modelo en cada modo."""
    if mode == "no_documents":
        return []
    if mode == "full_corpus":
        return chunks[CHUNK_SIZE]
    return [result.node for result in retrieve(CHUNK_SIZE, question["question"], TOP_K, CURRENT_ONLY_FILTER)]


def prompt_for(question, context):
    fragments = "\n\n".join(f"[{node.metadata['chunk_id']}] {node.metadata['title']}\n{node.get_content()}"
                             for node in context)
    return f"Fragmentos:\n{fragments or '(ninguno)'}\n\nPregunta: {question['question']}"


subset = [question for question in questions if question["in_generation_subset"]]
print(prompt_for(subset[0], context_for("retrieval", subset[0])))

In [ ]:
async def answer(question, mode):
    context = context_for(mode, question)
    model = create_model(model_ref, config, simulated_output=SIMULATED_ANSWER)
    agent = Agent(model, output_type=Answer, instructions=INSTRUCTION, model_settings=SETTINGS, retries=1)
    row = await timed_run(agent, prompt_for(question, context), ref=model_ref, config=config)
    return {"context_mode": mode, "id": question["id"], "attempts": 1,
            "context": {node.metadata["chunk_id"]: node.get_content() for node in context}, **row}


# Límites gratuitos (AI Studio): consultas y tokens por minuto. Con todo el corpus, Gemma admite pocas consultas
# por minuto, así que después de cada una se espera lo que pidan sus tokens.
LIMITS = {"google:gemma-4-26b-a4b-it": (30, 16_000), "google:gemini-3.5-flash-lite": (15, 250_000)}


def pause_after(row):
    if config.mode != "real" or model_ref not in LIMITS:
        return 0
    per_minute, tokens_per_minute = LIMITS[model_ref]
    return max(60 / per_minute, 60 * (row.get("input_tokens") or 0) / tokens_per_minute)


def service_failure(row):
    """Falló el servicio, no la consulta: se puede reintentar. Un 400 o un 401 volvería a fallar igual."""
    code = row.get("http_status")
    return row["status"] == "error" and (code is None or code == 429 or code >= 500)


answers = []
for question in subset:
    for mode in MODES:
        row = await answer(question, mode)
        answers.append(row)
        print(question["id"], mode, row["status"], row.get("error_type", ""), flush=True)
        await asyncio.sleep(pause_after(row))

failed = [index for index, row in enumerate(answers) if service_failure(row)]
if failed:
    print(f"Reintentando una vez {len(failed)} consultas fallidas en 20 s…")
    await asyncio.sleep(20)
for index in failed:
    answers[index] = {**await answer(by_id[answers[index]["id"]], answers[index]["context_mode"]), "attempts": 2}
    print(answers[index]["id"], answers[index]["context_mode"], answers[index]["status"], answers[index].get("error_type", ""))
print("Consultas con error:", sum(row["status"] == "error" for row in answers), "de", len(answers))

## 7. Comprobar las citas y la abstención

Ahora medimos el **generador** sin juez:

- **Cita verificable**: el fragmento citado estaba en el contexto que recibió el modelo y contiene la frase citada, letra por letra. Una cita verificable no prueba que la fuente sea la correcta: una cita literal de la política derogada pasa esta prueba y la respuesta es falsa.
- **Abstención correcta**: se abstuvo en las tres preguntas sin respuesta y respondió en las demás.
- **Evidencia esperada citada**: alguna cita contiene una de las frases esperadas, o está contenida en ella.

La corrección de la respuesta frente a la referencia la revisan ustedes: lean la tabla y elijan una **falla de generación** para el reporte.

In [ ]:
def citation_checks(row):
    if row["status"] != "ok":
        return {"citations": 0, "verifiable": 0, "cites_expected": False}
    citations = row["output"]["citations"]
    quotes = [evidence["quote"] for evidence in by_id[row["id"]]["expected_evidence"]]
    verifiable = [c["chunk_id"] in row["context"] and c["quote"] in row["context"][c["chunk_id"]] for c in citations]
    cites_expected = any(c["quote"] in quote or quote in c["quote"] for c in citations for quote in quotes)
    return {"citations": len(citations), "verifiable": sum(verifiable), "cites_expected": cites_expected}


generation = pd.DataFrame([{
    "id": row["id"], "context_mode": row["context_mode"], "status": row["status"],
    "answerable": by_id[row["id"]]["answerable"],
    "abstained": row["output"]["abstained"] if row["status"] == "ok" else None,
    **citation_checks(row),
    "context_chunks": len(row["context"]),
    "input_tokens": row.get("input_tokens"), "latency_ms": row.get("latency_ms"),
    "answer": row["output"]["answer"] if row["status"] == "ok" else "",
    "expected_answer": by_id[row["id"]]["expected_answer"],
} for row in answers])
generation["correct_abstention"] = generation["abstained"] == ~generation["answerable"]

summary = generation[generation["status"] == "ok"].groupby("context_mode").agg(
    answers=("id", "size"), correct_abstention=("correct_abstention", "sum"),
    citations=("citations", "sum"), verifiable=("verifiable", "sum"), cites_expected=("cites_expected", "sum"),
    median_input_tokens=("input_tokens", "median"), median_latency_ms=("latency_ms", "median"),
).reindex(MODES)
display(summary)
with pd.option_context("display.max_colwidth", None):
    display(generation[["id", "context_mode", "abstained", "citations", "verifiable", "answer", "expected_answer"]])

## 8. Un solo cambio: reordenar con un *reranker*

Un ***reranker*** relee cada par pregunta-fragmento con un modelo más preciso y más lento que la comparación de vectores, y los reordena. Lo usamos como **un único cambio** sobre la configuración elegida: recuperamos 20 candidatos, los reordena `gte-multilingual-reranker-base` y conservamos los *k* primeros. Si las métricas cambian, sabemos a qué atribuirlo.

Corre en tu sesión de Colab, sin clave: la primera vez descarga el modelo (unos 600 MB). Con **CPU** puede tardar unos minutos; con **GPU T4** (Entorno de ejecución → Cambiar tipo de entorno) es mucho más rápido. Anota el tiempo y el tipo de entorno. Con fragmentos de 400 el corpus tiene menos de 20 fragmentos, así que el *reranker* reordena el corpus completo.

In [ ]:
from llama_index.core.postprocessor import SentenceTransformerRerank

RUN_RERANK = True  # @param {type: "boolean"}
RERANK_MODEL = "Alibaba-NLP/gte-multilingual-reranker-base"
CANDIDATES = 20

if RUN_RERANK:
    start = perf_counter()
    reranker = SentenceTransformerRerank(model=RERANK_MODEL, top_n=TOP_K)
    load_s = perf_counter() - start
    start, reranked = perf_counter(), []
    for question in questions:
        candidates = retrieve(CHUNK_SIZE, question["question"], CANDIDATES, CURRENT_ONLY_FILTER)
        top = reranker.postprocess_nodes(candidates, query_str=question["question"])
        reranked.append({"id": question["id"], **retrieval_scores(question, top)})
    rerank_s = perf_counter() - start
    device = reranker.device
    print(f"Dispositivo: {device} · carga del modelo: {load_s:.0f} s · 24 preguntas: {rerank_s:.1f} s")
    sweep.append({"chunk_size": CHUNK_SIZE, "chunks": len(chunks[CHUNK_SIZE]), "k": TOP_K,
                  "current_only": CURRENT_ONLY_FILTER, "rerank": True, **summarize(reranked),
                  "rerank_device": device, "rerank_seconds": round(rerank_s, 1)})
    metrics = pd.DataFrame(sweep)
    display(metrics[(metrics["chunk_size"] == CHUNK_SIZE) & (metrics["k"] == TOP_K)
                    & (metrics["current_only"] == CURRENT_ONLY_FILTER)])

## 9. Contraste con jueces de DeepEval (sin nota)

Un **juez** es un modelo de lenguaje que evalúa la salida de otro. DeepEval ofrece dos métricas que separan los componentes:

- **Faithfulness** (fidelidad): ¿las afirmaciones de la respuesta se sostienen en los fragmentos recuperados? Mide al generador.
- **Contextual Recall** (recall de contexto): ¿los fragmentos recuperados contienen lo que dice la respuesta de referencia? Mide al recuperador.

Usamos dos jueces, Gemini y el servidor del curso (este último solo en clase), sobre las respuestas del modo `retrieval`. El juez **no reemplaza** a las métricas deterministas de las secciones 4 y 7 ni cuenta para la nota: sirve para discutir los casos en que discrepa de ellas. Tope: 30 evaluaciones por juez y por equipo; aquí son 17 por juez (10 de fidelidad y 7 de recall, porque las preguntas sin respuesta no tienen referencia que recuperar). Cada evaluación hace dos o tres consultas al juez: con Gemini tarda unos minutos por la espera entre consultas.

In [ ]:
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"  # sin envío de datos de uso a DeepEval
from deepeval.metrics import ContextualRecallMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

from curso_ia.rag import judge_model

# Juez: (modelo, consultas por minuto). El servidor del curso no necesita espera.
JUDGES = {"gemini": ("google:gemini-3.5-flash-lite", 14), "server": ("curso:course-chat", None)}
MAX_EVALUATIONS = 30

judged = []
if config.mode == "simulated":
    print("Modo simulado: los jueces necesitan modelos reales; ejecuta en modo real para esta sección.")
else:
    for name, (ref, per_minute) in JUDGES.items():
        try:
            judge = judge_model(ref, config, per_minute=per_minute, settings={"temperature": 0})
        except Exception as error:
            print(f"Juez {name} ({ref}) no disponible: {type(error).__name__}. Se omite.")
            continue
        evaluations = 0
        for row in answers:
            if row["context_mode"] != "retrieval" or row["status"] != "ok":
                continue
            question = by_id[row["id"]]
            case = LLMTestCase(input=question["question"], actual_output=row["output"]["answer"],
                               expected_output=question["expected_answer"],
                               retrieval_context=list(row["context"].values()))
            metrics_for_case = [FaithfulnessMetric(model=judge, include_reason=False)]
            if question["answerable"]:
                metrics_for_case.append(ContextualRecallMetric(model=judge, include_reason=False))
            for metric in metrics_for_case:
                if evaluations >= MAX_EVALUATIONS:
                    break
                evaluations += 1
                try:
                    await metric.a_measure(case, _show_indicator=False)
                    judged.append({"judge": name, "model": ref, "id": row["id"],
                                   "metric": metric.__name__, "score": round(metric.score, 2)})
                except Exception as error:
                    judged.append({"judge": name, "model": ref, "id": row["id"], "metric": metric.__name__,
                                   "score": None, "error_type": type(error).__name__})
            print(name, row["id"], "listo", flush=True)
        print(f"Juez {name}: {evaluations} evaluaciones, {judge.requests} consultas, "
              f"{judge.input_tokens} tokens de entrada")

if judged:
    # Puntajes de los jueces junto a las métricas deterministas de la misma pregunta.
    scores = pd.DataFrame(judged).pivot_table(index="id", columns=["metric", "judge"], values="score")
    scores.columns = [f"{metric} · {judge}" for metric, judge in scores.columns]
    retrieval_checks = generation[generation["context_mode"] == "retrieval"].set_index("id")
    recall_chosen = chosen.set_index("id")["recall"].rename("recall_at_k")
    display(scores.join(retrieval_checks[["answerable", "abstained", "citations", "verifiable"]]).join(recall_chosen))

## 10. Guardar la evidencia y decidir

Guardamos el barrido en `metricas.csv` y, en `experiment.json`, la configuración elegida, la instrucción, el esquema, las 30 respuestas con el contexto que recibió cada una, el contraste de los jueces y sus decisiones. Con esos archivos otra persona puede verificar las citas y recalcular las métricas sin consultar a ningún modelo.

Completa `DECISIONS` antes de guardar:

1. **Configuración**: cuál eligen para su sistema y qué fila de `metricas.csv` lo justifica (evidencia encontrada, texto enviado, vigencia).
2. **Falla de recuperación**: una pregunta en que la evidencia no llegó o llegó la versión derogada, con su fila.
3. **Falla de generación**: una respuesta con la evidencia disponible que igual quedó mal (cita no verificable, cita de la fuente equivocada, abstención indebida o dato distinto de la referencia), con la cita que lo prueba.
4. **Juez**: un caso en que el juez discrepa de la métrica determinista y a cuál le creen.

Si no aparece alguna falla real, informa ese resultado; no inventes errores ni presentes salidas simuladas como mediciones.

In [ ]:
DECISIONS = {
    "configuration": "",       # Qué configuración eligen y qué fila la justifica
    "retrieval_failure": "",   # Una falla de recuperación con su evidencia
    "generation_failure": "",  # Una falla de generación con la cita que la prueba
    "judge_disagreement": "",  # Un caso en que el juez discrepa de la métrica determinista
}

OUTPUT.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT / "metricas.csv", index=False)
payload = {
    "models": {"generation": model_ref, "embedding": config.models["embedding"]},
    "precomputed_embeddings": PRECOMPUTED,
    "embedding_requests": embedding.requests,
    "chosen": {"chunk_size": CHUNK_SIZE, "top_k": TOP_K, "current_only": CURRENT_ONLY_FILTER},
    "settings": SETTINGS,
    "instruction": INSTRUCTION,
    "output_schema": Answer.model_json_schema(),
    "reference_notes": REFERENCE_NOTES,
    "retrieval_per_question": per_question,
    "answers": answers,
    "judges": judged,
    "decisions": DECISIONS,
}
save_experiment(OUTPUT / "experiment.json", payload, config, lab=3)
print("Evidencia guardada en", OUTPUT)

## 11. Descargar y entregar

Ejecuta la celda siguiente para descargar un ZIP con `metricas.csv` y `experiment.json`. Guarda también el notebook **con sus salidas** mediante **Archivo → Descargar → Descargar .ipynb**. Si haces otra iteración, descarga antes la versión actual: cada ejecución sobrescribe los archivos.

**Producto del equipo:** notebook, `metricas.csv`, `experiment.json` y las decisiones. Después se incorporan a `labs/lab03-rag/` del repositorio del equipo; solo lo validado se promueve a `src/`. No hace falta usar Git desde Colab.

**Cada integrante:** reflexión propia, historial completo del asistente usado o declaración de que no usó uno, y enlace al producto compartido. Retira credenciales y datos sensibles e indica las omisiones.

**Canvas:** responde la pregunta individual de cinco minutos al cierre de la clase del 30/10. Producto, reflexión e historial se entregan hasta el **martes 03/11 a las 23:59, hora de Chile**.

**Extensiones opcionales:**

- Las 18 configuraciones completas: agrega *k* = 10 al barrido de la sección 4 y el rerank a cada combinación.
- Preguntas propias: escribe preguntas sobre los documentos de su proyecto, con las frases literales esperadas, en la misma forma que `questions.json`; alimentan las 20 preguntas con fuentes del Hito 1. Cada pregunta nueva calcula su vector con tu clave.
- Embeddings locales: reemplaza `embedding` por un modelo de `sentence-transformers` y compara el barrido.

Referencias opcionales (inglés): [LlamaIndex: SentenceSplitter](https://docs.llamaindex.ai/en/stable/module_guides/loading/node_parsers/modules/), [Faithfulness](https://deepeval.com/docs/metrics-faithfulness) y [Contextual Recall](https://deepeval.com/docs/metrics-contextual-recall) de DeepEval, unos 10 minutos cada una.

In [ ]:
import shutil
from datetime import datetime, timezone

# Un nombre distinto conserva las iteraciones descargadas antes de sobrescribirlas.
archive_name = "lab03-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
archive_path = Path(shutil.make_archive(str(ROOT / archive_name), "zip", root_dir=ROOT / "labs", base_dir="lab03-rag"))
print("Descarga preparada:", archive_path.name)
try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab, copia el ZIP desde la ruta indicada.")
else:
    files.download(str(archive_path))